# Example 6b: 阶段分离 — 独立提取 ODB（新文件/新会话）

这个文件**不重新求解**，只做 `run_extraction()`：假设 `01_run_job.ipynb`
已经在别处把 job 跑完，这里单独打开一个新的 Python 会话，重新构造同样的
`BatchAbaqusProcessor`，直接读取已有的 ODB 并提取结果。

In [ ]:
import os

from ABQflow import BatchAbaqusProcessor, JobSpec, PreparationSpec, HookSpec

%reload_ext autoreload
%autoreload 2

ABAQUS_CAE = 'C:/Applications/SIMULIA/Commands/2026/abaqus.bat'
CWD = os.getcwd()
OUTPUT_DIR = os.path.join(CWD, "examples/06_SeparateJob/output")

## 重新声明与 `01_run_job.ipynb` 完全一致的 specs

`run_extraction()` 不会调用 `plan()`/`prepare()`，也不会执行 `preparation` 里的策略 ——
它只是把 `job_name` 拼回 `base_output_dir/job_name` 去找已有的 ODB。所以这里的
`preparation` 字段只需要结构上合法（`JobSpec` 要求 modular workflow 必须有
`preparation`），内容是否和当初生成 INP 时完全一致并不影响提取结果。真正要保持一致
的是 `job_name`、`base_output_dir`，以及要执行的 `post_extraction` 钩子。

In [2]:
YOUNGS_MODULUS_LIST = [190000, 200000, 210000]

specs = [
	JobSpec(
		job_name = f"separate_job_{i:02d}",
		workflow = "modular",
		preparation = PreparationSpec(
			kind = "inp_based",
			source_path = "./examples/cae_file/planar_stress_template.inp",
			params = {
				"youngs_modulus": e,
				"load_magnitude": 2000,
			}
		),
		post_extraction = [
			HookSpec(
				script_path = "./examples/extraction_scripts/get_max_stress_mises.py",
				tasks = [
					{"result_name": "max_stress_mises",},
					{"result_name": "max_displacement",},
				]
			)
		]
	)
	for i, e in enumerate(YOUNGS_MODULUS_LIST, start=1)
]

In [3]:
processor = BatchAbaqusProcessor(
	batch_data = specs,
	base_output_dir = OUTPUT_DIR,
	cpus_per_job = 4,
	abaqus_exe = ABAQUS_CAE,
	duplicate_mode = "skip",  # run_extraction() 用不到 plan()/prepare()，这里只是为了保险：
	                          # 万一手滑调用了 prepare() 也不会因为目录已存在而报错
)

## 阶段 3：只跑后处理提取（`run_extraction`）

假设 `ctx.odb_path` 已经存在（由 `01_run_job.ipynb` 产出）。

In [4]:
ext_outcomes = processor.run_extraction(num_parallel_jobs=3)

for oc in ext_outcomes:
	print(oc.job_name, oc.status)
	if oc.status == "COMPLETED":
		print("  max_stress_mises =", oc.results["max_stress_mises"])
		print("  max_displacement =", oc.results["max_displacement"])
	else:
		print("  error:", oc.error)

Output()

separate_job_02 COMPLETED
  max_stress_mises = 4525.26025390625
  max_displacement = 4.189039707183838
separate_job_01 COMPLETED
  max_stress_mises = 4525.26025390625
  max_displacement = 4.409515380859375
separate_job_03 COMPLETED
  max_stress_mises = 4525.26025390625
  max_displacement = 3.9895615577697754


## 小结

`01_run_job.ipynb` 和 `02_extract_odb.ipynb` 是两个完全独立的 Python 进程/会话：
中间可以关掉 Jupyter、重启机器，只要 `base_output_dir` 里的 ODB 还在，
`run_extraction()` 就能在任意时间、任意新会话里重新提取结果 —— 这正是阶段分离要解决的问题。